# Original MSc dissertation notebook (August 2024)

Kept for transparency, with outputs cleared. **The maintained, tested version of this work is in `src/` and `scripts/`; use its results (see the README), not this notebook's.**

Known issues in this notebook, all fixed in `src/`:

1. The **"Solar RFR Model" cell reads `wind_merged_data`**, so the dissertation's solar Random Forest row is the wind result repeated.
2. Production timestamps were rebuilt from `Date` + `StartHour`, which **drops the UTC offset** and misaligns production with the UTC weather by one to two hours. This depressed the solar scores most.
3. Models were scored on **different protocols**: Random Forest, SVR and ARIMA on a shuffled random split, LSTM and Prophet on a chronological split, so the model comparison was not like-for-like.
4. **Holt-Winters used an additive trend extrapolated over the whole test horizon**, which diverges and explains its very large errors.

In [ ]:
# Dependencies
import urllib.request
import csv
import codecs
import pandas as pd
from datetime import datetime, timedelta
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Downloading the kaggle dataset into a pandas dataframe

production_data = pd.read_csv("intermittent-renewables-production-france.csv")
production_data.dtypes

In [ ]:
# Value counts
print(production_data['Source'].value_counts())

# Bar chart
production_data['Source'].value_counts().plot(kind='bar')
plt.xlabel('Source')
plt.ylabel('Frequency')
plt.title('Distribution of Source Values')
plt.show()



In [ ]:
solar_data = production_data[production_data['Source'] == 'Solar']
wind_data = production_data[production_data['Source'] == 'Wind']


def timestamp(data):
    data['timestamp'] = pd.to_datetime(data['Date'] + ' ' + data['StartHour'], utc=True)
    data = data.drop_duplicates(subset=['timestamp'])
    return data

solar_data = timestamp(solar_data)
wind_data = timestamp(wind_data)


In [ ]:
# Downloading weather merged_data from https://power.larc.nasa.gov/merged_data-access-viewer/ to access
# meterological merged_data for a central point in France

weather_data = pd.read_csv("POWER_Point_Hourly_20200622_20230630_047d27N_002d36E_LST.csv", skiprows = 15)

In [ ]:
weather_data.head()

In [ ]:
wind_data

In [ ]:
# # Since the timestamps for both these datasets is different we are making them both into a similar format. Then we are
# # Comparing the two datasets to see if there are any missing timestamps.

# # Convert date and time columns to a single datetime column for production data
# production_data['timestamp'] = pd.to_datetime(production_data['Date'] + ' ' + production_data['StartHour'], utc=True)

# Convert year, month, day, and hour columns to a single datetime column for weather data
weather_data['timestamp'] = pd.to_datetime(weather_data['YEAR'].astype(str) + '-' +
                                           weather_data['MO'].astype(str) + '-' +
                                           weather_data['DY'].astype(str) + ' ' +
                                           weather_data['HR'].astype(str) + ':00', utc=True)

# # Check for missing timestamps in production data
# missing_prod_timestamps = pd.date_range(start=production_data['timestamp'].min(),
#                                         end=production_data['timestamp'].max(),
#                                         freq='H').difference(production_data['timestamp'])

# Check for missing timestamps in weather data
missing_weather_timestamps = pd.date_range(start=weather_data['timestamp'].min(),
                                           end=weather_data['timestamp'].max(),
                                           ).difference(weather_data['timestamp'])

# # Print missing timestamps
# print("Missing production data timestamps:", missing_prod_timestamps)
print("Missing weather merged_data timestamps:", missing_weather_timestamps)

In [ ]:
# Check for missing timestamps in production data
missing_solar_timestamps = pd.date_range(start=solar_data['timestamp'].min(),
                                        end=solar_data['timestamp'].max(),
                                        freq='H').difference(production_data['timestamp'])

# Check for missing timestamps in weather data
missing_wind_timestamps = pd.date_range(start=wind_data['timestamp'].min(),
                                           end=wind_data['timestamp'].max(),
                                           ).difference(weather_data['timestamp'])

# Print missing timestamps
print("Missing production data timestamps:", missing_solar_timestamps)
print("Missing weather merged_data timestamps:", missing_wind_timestamps)


### Merging datasets without interpolation (tested using interpolation as well)

In [ ]:
# This is the total size of the merged data. Keep in mind this data needs to be cleaned up to remove duplicate
# timestamps

# production_data['timestamp'] = pd.to_datetime(production_data['Date'] + ' ' + production_data['StartHour'], utc=True)


# weather_data['timestamp'] = pd.to_datetime(weather_data['YEAR'].astype(str) + '-' +
#                                            weather_data['MO'].astype(str) + '-' +
#                                            weather_data['DY'].astype(str) + ' ' +
#                                            weather_data['HR'].astype(str) + ':00', utc=True)

# production_data = production_data.drop_duplicates(subset=['timestamp'])

# production_data.set_index('timestamp', inplace=True)
# production_data.reset_index(inplace=True)

# weather_data.set_index('timestamp', inplace=True)
# weather_data.reset_index(inplace=True)

# merged_data = pd.merge(production_data, weather_data, on='timestamp', how='inner')
# merged_data.drop(['Date', 'StartHour', 'EndHour', 'Source'], axis=1, inplace=True)

# print("Size of merged data:", merged_data.shape)

In [ ]:
def merge_with_weather(production_data, weather_data):
    merged_data = pd.merge(production_data, weather_data, on='timestamp', how='inner')
    merged_data.drop(['Date', 'StartHour', 'EndHour', 'Source'], axis=1, inplace=True)
    merged_data.set_index('timestamp', inplace=True)
    return merged_data

solar_merged_data = merge_with_weather(solar_data, weather_data)
wind_merged_data = merge_with_weather(wind_data, weather_data)

print("Size of merged solar data:", solar_merged_data.shape)
print("Size of merged wind data:", wind_merged_data.shape)

In [ ]:
weather_data

In [ ]:
wind_merged_data

A lot of values in the merged_interpol and merged_data for the feature ALLSKY_KT is -999.00 which seemed to be a
NaN value so I decided to look further into what this weather feature means.

KT to ALLSKY_KT: All Sky Insolation Clearness Index - A fraction representing clearness of the atmosphere and the all sky insolation that is transmitted through the atmosphere to strike the surface of the earth divided by the average of top of the atmosphere total solar irradiance incidents.

In [ ]:
# Handle missing values

# If running the cells from the beginning run the convert timestamp to datetime only once otherwise the code keeps
# looking for a column that doesnt exist.

wind_merged_data.replace(-999.00, float('nan'), inplace=True)
solar_merged_data.replace(-999.00, float('nan'), inplace=True)


# merged_data_fill = merged_data.fillna(merged_data.mean(), inplace=True)
# merged_data_na = merged_data.dropna()

# # Convert timestamp to datetime
# merged_data['timestamp'] = pd.to_datetime(merged_data['timestamp'])

# # Set timestamp as index
# merged_data.reset_index(inplace = True)

In [ ]:
wind_merged_data.isnull().sum()

In [ ]:
solar_merged_data.isnull().sum()

In [ ]:
solar_merged_data = solar_merged_data.dropna(subset = ['Production'])
wind_merged_data = wind_merged_data.dropna(subset = ['Production'])

In [ ]:
# Correlation matrix for solar data
import seaborn as sns
import matplotlib.pyplot as plt

solar_corr_matrix = solar_merged_data.corr().round(2)
print("Solar Correlation Matrix")


# Plotting the correlation matrix for solar data
plt.figure(figsize=(10, 8))
sns.heatmap(solar_corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix for Solar Data')
plt.savefig('solar_correlation_matrix.png')
plt.show()

# Correlation matrix for wind data
wind_corr_matrix = wind_merged_data.corr().round(2)
print("Wind Correlation Matrix")


# Plotting the correlation matrix for wind data
plt.figure(figsize=(10, 8))
sns.heatmap(wind_corr_matrix, annot=True, cmap='coolwarm')
plt.title('Correlation Matrix for Wind Data')
plt.savefig('wind_correlation_matrix.png')
plt.show()


In [ ]:
solar_merged_data.reset_index(inplace=True)
wind_merged_data.reset_index(inplace=True)

In [ ]:
import seaborn as sns



# Plot Time Series of Production"
plt.figure(figsize=(14, 6))
plt.plot(solar_merged_data.index, solar_merged_data['Production'], label='Production')
plt.xlabel('Date')
plt.ylabel('Production')
plt.title('Time Series of Solar Production')
plt.legend()
plt.show()

# Plot Time Series of Production"
plt.figure(figsize=(14, 6))
plt.plot(wind_merged_data.index, solar_merged_data['Production'], label='Production')
plt.xlabel('Date')
plt.ylabel('Production')
plt.title('Time Series of Wind Production')
plt.legend()
plt.show()

In [ ]:

wind_merged_data.set_index('timestamp', inplace=True)
solar_merged_data.set_index('timestamp', inplace=True)

# Plot and save the distribution of Solar Production
plt.figure(figsize=(10, 6))
sns.histplot(solar_merged_data['Production'], bins=30, kde=True)
plt.xlabel('Production')
plt.ylabel('Frequency')
plt.title('Distribution of Solar Production')
plt.savefig('solar_production_distribution.png')  # Save the figure as a PNG file
plt.show()

# Plot and save the distribution of Wind Production
plt.figure(figsize=(10, 6))
sns.histplot(wind_merged_data['Production'], bins=30, kde=True)
plt.xlabel('Production')
plt.ylabel('Frequency')
plt.title('Distribution of Wind Production')
plt.savefig('wind_production_distribution.png')  # Save the figure as a PNG file
plt.show()


In [ ]:
solar_merged_data.describe()
wind_merged_data.describe()

In [ ]:
# # Correlation Heatmap
# plt.figure(figsize=(12, 8))
# correlation = merged_data.corr()
# sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt='.2f')
# plt.title('Correlation Heatmap')
# plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=6, ncols=1, figsize=(17, 25), sharex=True)

solar_merged_data['ALLSKY_SFC_SW_DWN'].plot(ax=axes[0], title='All Sky Surface Shortwave Downward (ALLSKY_SFC_SW_DWN)', color='g')
axes[0].set_ylabel('SW Downward')

solar_merged_data['ALLSKY_SFC_SW_DNI'].plot(ax=axes[1], title='All Sky Surface Direct Normal Irradiance (ALLSKY_SFC_SW_DNI)', color='c')
axes[1].set_ylabel('SW DNI')

solar_merged_data['ALLSKY_SFC_SW_DIFF'].plot(ax=axes[2], title='All Sky Surface Shortwave Diffuse (ALLSKY_SFC_SW_DIFF)', color='m')
axes[2].set_ylabel('SW Diffuse')

solar_merged_data['ALLSKY_KT'].plot(ax=axes[3], title='All Sky Clearness Index (ALLSKY_KT)', color='y')
axes[3].set_ylabel('Clearness Index')

solar_merged_data['T2M'].plot(ax=axes[4], title='Temperature (T2M)', color='r')
axes[4].set_ylabel('Temperature (°C)')

solar_merged_data['WS10M'].plot(ax=axes[5], title='Wind Speed (WS10M)', color='b')
axes[5].set_ylabel('Wind Speed (m/s)')

plt.xlabel('Date and Time')

# Save the figure as a PNG file
plt.savefig('solar_data_plots.png')

# Show the plot
plt.show()

In [ ]:

fig, axes = plt.subplots(nrows=6, ncols=1, figsize=(17, 25), sharex=True)

wind_merged_data['ALLSKY_SFC_SW_DWN'].plot(ax=axes[0], title='All Sky Surface Shortwave Downward (ALLSKY_SFC_SW_DWN)', color='g')
axes[0].set_ylabel('SW Downward')

wind_merged_data['ALLSKY_SFC_SW_DNI'].plot(ax=axes[1], title='All Sky Surface Direct Normal Irradiance (ALLSKY_SFC_SW_DNI)', color='c')
axes[1].set_ylabel('SW DNI')

wind_merged_data['ALLSKY_SFC_SW_DIFF'].plot(ax=axes[2], title='All Sky Surface Shortwave Diffuse (ALLSKY_SFC_SW_DIFF)', color='m')
axes[2].set_ylabel('SW Diffuse')

wind_merged_data['ALLSKY_KT'].plot(ax=axes[3], title='All Sky Clearness Index (ALLSKY_KT)', color='y')
axes[3].set_ylabel('Clearness Index')

wind_merged_data['T2M'].plot(ax=axes[4], title='Temperature (T2M)', color='r')
axes[4].set_ylabel('Temperature (°C)')

wind_merged_data['WS10M'].plot(ax=axes[5], title='Wind Speed (WS10M)', color='b')
axes[5].set_ylabel('Wind Speed (m/s)')

plt.xlabel('Date and Time')

# Save the figure as a PNG file
plt.savefig('wind_data_plots.png')

# Show the plot
plt.show()


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import GridSearchCV

### Wind Random Forest Model

In [ ]:



features = ['ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_SW_DNI', 'ALLSKY_SFC_SW_DIFF', 'T2M', 'WS10M']
target = 'Production'

# Split data into training and testing sets
X = wind_merged_data[features]
y = wind_merged_data[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

# Normalize/scale the merged data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


param_grid = {
    'n_estimators': [500, 1000],
    'max_depth': [10],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 4],
    'bootstrap': [True, False]
}


# Initialize the Random Forest model
rf = RandomForestRegressor(random_state=42)

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid,
                           cv=2, n_jobs=1, verbose=2, scoring='neg_mean_squared_error')

# Fit the GridSearchCV to the data
grid_search.fit(X_train_scaled, y_train)

# Extract the best model
best_rf = grid_search.best_estimator_

# Make predictions with the optimized model
y_pred_best_rf = best_rf.predict(X_test_scaled)

# Calculate metrics for the optimized model
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best_rf))
mae_best = mean_absolute_error(y_test, y_pred_best_rf)
r2_best = r2_score(y_test, y_pred_best_rf)
mse_best = mean_squared_error(y_test, y_pred_best_rf)

print(f"Optimized RMSE: {rmse_best:.4f}, MAE: {mae_best:.4f}, R² Score: {r2_best:.4f}, MSE: {mse_best:.4f}")


plt.figure(figsize=(10, 6))
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Actual')
plt.scatter(y_test, y_pred_best_rf, color='blue', alpha=0.5, label='Predicted')
plt.xlabel('Actual Production')
plt.ylabel('Predicted Production')
plt.title('Regression Line Plot of Actual vs Predicted Production (Optimized)')
plt.legend()
plt.show()

residuals_best = y_test - y_pred_best_rf

# Histogram of residuals
plt.figure(figsize=(10, 6))
plt.hist(residuals_best, bins=30, edgecolor='k')
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title('Distribution of Residuals (Optimized)')
plt.show()

plt.figure(figsize=(10, 6))
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.scatter(y_test, y_pred_best_rf, color='blue', alpha=0.5)
plt.xlabel('Actual Production')
plt.ylabel('Predicted Production')
plt.title('Regression Line Plot of Actual vs Predicted Production (Optimized)')
plt.show()

# Print the best parameters
print("Best parameters found by GridSearchCV:")
print(grid_search.best_params_)


### Solar RFR Model

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


features = ['ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_SW_DNI', 'ALLSKY_SFC_SW_DIFF', 'T2M', 'WS10M']
target = 'Production'

# Split data into training and testing sets
X = wind_merged_data[features]
y = wind_merged_data[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

# Normalize/scale the merged data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define the parameter grid for GridSearchCV
param_grid = {
    'n_estimators': [500, 1000],
    'max_depth': [10],
    'min_samples_split': [2, 10],
    'min_samples_leaf': [1, 4],
    'bootstrap': [True, False]
}

# Initialize the Random Forest model
rf = RandomForestRegressor(random_state=42)

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=rf, param_grid=param_grid,
                           cv=2, n_jobs=1, verbose=2, scoring='neg_mean_squared_error')

# Fit the GridSearchCV to the data
grid_search.fit(X_train_scaled, y_train)

# Extract the best model
best_rf = grid_search.best_estimator_

# Make predictions with the optimized model
y_pred_best_rf = best_rf.predict(X_test_scaled)

# Calculate metrics for the optimized model
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best_rf))
mae_best = mean_absolute_error(y_test, y_pred_best_rf)
r2_best = r2_score(y_test, y_pred_best_rf)
mse_best = mean_squared_error(y_test, y_pred_best_rf)

print(f"Optimized RMSE: {rmse_best:.4f}, MAE: {mae_best:.4f}, R² Score: {r2_best:.4f}, MSE: {mse_best:.4f}")

plt.figure(figsize=(10, 6))
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Actual')
plt.scatter(y_test, y_pred_best_rf, color='blue', alpha=0.5, label='Predicted')
plt.xlabel('Actual Production')
plt.ylabel('Predicted Production')
plt.title('Regression Line Plot of Actual vs Predicted Production (Optimized)')
plt.legend()
plt.show()


residuals_best = y_test - y_pred_best_rf

# Histogram of residuals
plt.figure(figsize=(10, 6))
plt.hist(residuals_best, bins=30, edgecolor='k')
plt.xlabel('Residuals')
plt.ylabel('Frequency')
plt.title('Distribution of Residuals (Optimized)')
plt.show()

plt.figure(figsize=(10, 6))
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.scatter(y_test, y_pred_best_rf, color='blue', alpha=0.5)
plt.xlabel('Actual Production')
plt.ylabel('Predicted Production')
plt.title('Regression Line Plot of Actual vs Predicted Production (Optimized)')
plt.show()

# Print the best parameters
print("Best parameters found by GridSearchCV:")
print(grid_search.best_params_)

### Wind SVR Model

In [ ]:
from sklearn.svm import SVR
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

features = ['ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_SW_DNI', 'ALLSKY_SFC_SW_DIFF', 'T2M', 'WS10M']
target = 'Production'

# Split data into training and testing sets
X = wind_merged_data[features]
y = wind_merged_data[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

# Normalize/scale the merged data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define the parameter grid for GridSearchCV
param_grid = {
    'C': [100, 1000],
    'gamma': ['scale', 'auto'],
    'epsilon': [0.1, 0.2, 0.5],
    'kernel': ['rbf']
}

# Initialize the SVM model
svm = SVR()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=svm, param_grid=param_grid,
                           cv=3, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error')

# Fit the GridSearchCV to the data
grid_search.fit(X_train_scaled, y_train)

# Extract the best model
best_svm = grid_search.best_estimator_

# Make predictions with the optimized model
y_pred_best_svm = best_svm.predict(X_test_scaled)

# Calculate metrics for the optimized model
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best_svm))
mae_best = mean_absolute_error(y_test, y_pred_best_svm)
r2_best = r2_score(y_test, y_pred_best_svm)
mse_best = mean_squared_error(y_test, y_pred_best_svm)

print(f"Optimized RMSE: {rmse_best:.4f}, MAE: {mae_best:.4f}, R² Score: {r2_best:.4f}, MSE: {mse_best:.4f}")

plt.figure(figsize=(10, 6))
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Actual')
plt.scatter(y_test, y_pred_best_svm, color='blue', alpha=0.5, label='Predicted')
plt.xlabel('Actual Production')
plt.ylabel('Predicted Production')
plt.title('Regression Line Plot of Actual vs Predicted Production')
plt.legend()
plt.show()

# Distribution of residuals
residuals_best = y_test - y_pred_best_svm
plt.figure(figsize=(10, 6))
plt.hist(residuals_best, bins=30, edgecolor='k')
plt.xlabel('Residuals (SVM)')
plt.ylabel('Frequency')
plt.title('Distribution of Residuals (Optimized SVM)')
plt.show()

# Regression line plot
plt.figure(figsize=(10, 6))
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.scatter(y_test, y_pred_best_svm, color='green', alpha=0.5)
plt.xlabel('Actual Production')
plt.ylabel('Predicted Production (SVM)')
plt.title('Regression Line Plot of Actual vs Predicted Production (Optimized SVM)')
plt.show()

# Print the best parameters
print("Best parameters found by GridSearchCV:")
print(grid_search.best_params_)


### Solar SVR Model

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR

features = ['ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_SW_DNI', 'ALLSKY_SFC_SW_DIFF', 'T2M', 'WS10M']
target = 'Production'

# Split data into training and testing sets
X = solar_merged_data[features]
y = solar_merged_data[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

# Normalize/scale the merged data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Define the parameter grid for GridSearchCV
param_grid = {
    'C': [100, 1000],
    'gamma': ['scale', 'auto'],
    'epsilon': [0.2, 0.5],
    'kernel': ['rbf']
}

# Initialize the SVM model
svm = SVR()

# Set up GridSearchCV
grid_search = GridSearchCV(estimator=svm, param_grid=param_grid,
                           cv=3, n_jobs=-1, verbose=2, scoring='neg_mean_squared_error')

# Fit the GridSearchCV to the data
grid_search.fit(X_train_scaled, y_train)

# Extract the best model
best_svm = grid_search.best_estimator_

# Make predictions with the optimized model
y_pred_best_svm = best_svm.predict(X_test_scaled)

# Calculate metrics for the optimized model
rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best_svm))
mae_best = mean_absolute_error(y_test, y_pred_best_svm)
r2_best = r2_score(y_test, y_pred_best_svm)
mse_best = mean_squared_error(y_test, y_pred_best_svm)

print(f"Optimized RMSE: {rmse_best:.4f}, MAE: {mae_best:.4f}, R² Score: {r2_best:.4f}, MSE: {mse_best:.4f}")

plt.figure(figsize=(10, 6))
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Actual')
plt.scatter(y_test, y_pred_best_svm, color='blue', alpha=0.5, label='Predicted')
plt.xlabel('Actual Production')
plt.ylabel('Predicted Production')
plt.title('Regression Line Plot of Actual vs Predicted Production')
plt.legend()
plt.show()


# Distribution of residuals
residuals_best = y_test - y_pred_best_svm
plt.figure(figsize=(10, 6))
plt.hist(residuals_best, bins=30, edgecolor='k')
plt.xlabel('Residuals (SVM)')
plt.ylabel('Frequency')
plt.title('Distribution of Residuals (Optimized SVM)')
plt.show()

# Regression line plot
plt.figure(figsize=(10, 6))
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.scatter(y_test, y_pred_best_svm, color='green', alpha=0.5)
plt.xlabel('Actual Production')
plt.ylabel('Predicted Production (SVM)')
plt.title('Regression Line Plot of Actual vs Predicted Production (Optimized SVM)')
plt.show()

# Print the best parameters
print("Best parameters found by GridSearchCV:")
print(grid_search.best_params_)


### ARIMA Model

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
import numpy as np
import matplotlib.pyplot as plt
import itertools
from statsmodels.tsa.stattools import acf, pacf


# Assume y_train_array is stationary or has been differenced already if needed
y_train_array = y_train.to_numpy()

# ACF and PACF plots to find p and q
plt.figure(figsize=(12, 6))
plt.subplot(121)
plt.plot(acf(y_train_array, nlags=20))
plt.title('ACF Plot')
plt.subplot(122)
plt.plot(pacf(y_train_array, nlags=20))
plt.title('PACF Plot')
plt.show()

# Grid search for p, d, q using AIC
p = q = range(0, 3)
d = 0  # Assuming no differencing needed since the series is stationary
pdq = list(itertools.product(p, [d], q))

best_aic = np.inf
best_pdq = None

for param in pdq:
    try:
        model = ARIMA(y_train_array, order=param)
        results = model.fit()
        if results.aic < best_aic:
            best_aic = results.aic
            best_pdq = param
    except Exception as e:
        print(f"ARIMA order {param} failed with error: {e}")
        continue

print(f'Best ARIMA order: {best_pdq} with AIC: {best_aic}')


In [ ]:
wind_merged_data.reset_index()

In [ ]:
wind_merged_data.set_index('Date and Hour', inplace=True)

### Wind ARIMA Model

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, r2_score

wind_merged_data = wind_merged_data.dropna()

features =['WS10M']

X = wind_merged_data[features]
y = wind_merged_data[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train_array = y_train.to_numpy()
y_test_array = y_test.to_numpy()

# Fit the model with the best parameters
arima_model = ARIMA(y_train_array, order=(0,0,1))
arima_model_fit = arima_model.fit()

# Predict using the ARIMA model
y_pred_arima = arima_model_fit.forecast(steps=len(y_test_array))

plt.figure(figsize=(14, 5))
plt.plot(y_test_array.flatten(), label='Actual Production', color='blue')
plt.plot(y_pred_arima.flatten(), label='Predicted Production', color='orange')
plt.xlabel('Time')
plt.ylabel('Production')
plt.title('Actual vs Predicted Production (LSTM)')
plt.legend()
plt.grid(True)
plt.show()

# Scatter Plot of Actual vs. Predicted Values
plt.figure(figsize=(8, 8))
plt.scatter(y_test_array.flatten(), y_pred_arima.flatten(), alpha=0.5, color='purple')
plt.xlabel('Actual Production')
plt.ylabel('Predicted Production')
plt.title('Actual vs Predicted Production')
plt.plot([y_test_array.min(), y_test_array.max()], [y_test_array.min(), y_test_array.max()], 'r--')  # Line for perfect prediction
plt.grid(True)
plt.show()

# Plotting the Error Distribution
errors = y_test_array - y_pred_arima
plt.figure(figsize=(10, 6))
sns.histplot(errors.flatten(), kde=True, color='green')
plt.xlabel('Prediction Error')
plt.title('Error Distribution')
plt.grid(True)
plt.show()

rmse_best = np.sqrt(mean_squared_error(y_test_array, y_pred_arima))
mae_best = mean_absolute_error(y_test_array, y_pred_arima)
r2_best = r2_score(y_test_array, y_pred_arima)
mse_best = mean_squared_error(y_test_array, y_pred_arima)

print(f"Optimized RMSE: {rmse_best:.4f}, MAE: {mae_best:.4f}, R² Score: {r2_best:.4f}, MSE: {mse_best:.4f}")

### Solar ARIMA Model

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, r2_score

solar_merged_data = solar_merged_data.dropna()

features = ['ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_SW_DNI', 'ALLSKY_SFC_SW_DIFF', 'T2M']

X = solar_merged_data[features]
y = solar_merged_data[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

y_train_array = y_train.to_numpy()
y_test_array = y_test.to_numpy()


# Setting seaborn style for better aesthetics
sns.set_theme(style="whitegrid")

# Fit the ARIMA model with the best parameters
arima_model = ARIMA(y_train_array, order=(0,0,1))
arima_model_fit = arima_model.fit()

# Predict using the ARIMA model
y_pred_arima = arima_model_fit.forecast(steps=len(y_test_array))

# Plotting Actual vs Predicted Production over Time
plt.figure(figsize=(14, 6))
plt.plot(y_test_array.flatten(), label='Actual Production', color='blue', lw=2)
plt.plot(y_pred_arima.flatten(), label='Predicted Production', color='orange', linestyle='--', lw=2)
plt.xlabel('Time', fontsize=14)
plt.ylabel('Production (MWh)', fontsize=14)
plt.title('Actual vs Predicted Production (ARIMA)', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()

# Scatter Plot of Actual vs Predicted Production
plt.figure(figsize=(8, 8))
plt.scatter(y_test_array.flatten(), y_pred_arima.flatten(), alpha=0.6, color='purple', edgecolors='k')
plt.plot([y_test_array.min(), y_test_array.max()], [y_test_array.min(), y_test_array.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Production (MWh)', fontsize=14)
plt.ylabel('Predicted Production (MWh)', fontsize=14)
plt.title('Scatter Plot of Actual vs Predicted Production (ARIMA)', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()

# Plotting the Error Distribution
errors = y_test_array - y_pred_arima
plt.figure(figsize=(10, 6))
sns.histplot(errors.flatten(), kde=True, color='green', bins=30, edgecolor='black')
plt.xlabel('Prediction Error (MWh)', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
plt.title('Error Distribution (ARIMA)', fontsize=16)
plt.grid(True)
plt.show()

# Calculating and printing metrics
rmse_best = np.sqrt(mean_squared_error(y_test_array, y_pred_arima))
mae_best = mean_absolute_error(y_test_array, y_pred_arima)
r2_best = r2_score(y_test_array, y_pred_arima)
mse_best = mean_squared_error(y_test_array, y_pred_arima)

print(f"Optimized RMSE: {rmse_best:.4f}, MAE: {mae_best:.4f}, R² Score: {r2_best:.4f}, MSE: {mse_best:.4f}")

Look into lagged features and include in the literature or the methodology on why they were used. Also include literature on the stuff covered in class optimizers and use of dropout layers

### LSTM Model

In [ ]:
from sklearn.preprocessing import MinMaxScaler
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout
from sklearn.metrics import r2_score, mean_absolute_error
import matplotlib.pyplot as plt


In [ ]:
solar_merged_data.isnull().sum()

In [ ]:
wind_merged_data.isnull().sum()

### Wind LSTM Model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Assuming wind_merged_data is already loaded and available
features = ['ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_SW_DNI', 'ALLSKY_SFC_SW_DIFF', 'WS10M', 'T2M']
target = 'Production'

wind_merged_data = wind_merged_data.dropna()

X = wind_merged_data[features]
y = wind_merged_data[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, shuffle=False)

scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

def create_sequences(features, target, time_steps=12):
    X, y = [], []
    for i in range(len(features) - time_steps):
        X.append(features[i:i + time_steps])
        y.append(target[i + time_steps])
    return np.array(X), np.array(y)

time_steps = 12
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, time_steps)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, time_steps)

def build_model(hp):
    units = hp.Int('units', min_value=50, max_value=100, step=25)
    dropout_rate = hp.Float('dropout_rate', min_value=0.2, max_value=0.3, step=0.1)
    learning_rate = hp.Choice('learning_rate', values=[0.001, 0.01])

    model = Sequential([
        Input(shape=(X_train_seq.shape[1], X_train_seq.shape[2])),
        Bidirectional(LSTM(units, return_sequences=True)),
        Dropout(dropout_rate),
        Bidirectional(LSTM(units, return_sequences=False)),
        Dropout(dropout_rate),
        Dense(1)
    ])

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mean_squared_error')
    return model

# Keras Tuner
tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=10,
    executions_per_trial=1,
    directory='keras_tuner_dir',
    project_name='lstm_tuning'
)

early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.00001)

# Perform the hyperparameter search
tuner.search(X_train_seq, y_train_seq, epochs=15, validation_split=0.1,
             callbacks=[early_stopping, reduce_lr], verbose=2)

# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
The optimal number of units in the LSTM layer is {best_hps.get('units')},
the optimal dropout rate is {best_hps.get('dropout_rate')},
and the optimal learning rate is {best_hps.get('learning_rate')}.
""")

# Build the model with the optimal hyperparameters and train it
best_model = tuner.hypermodel.build(best_hps)

# Manually specify the number of epochs and batch size
history = best_model.fit(X_train_seq, y_train_seq, epochs=15,  # Set epochs here manually
                         batch_size=32,  # Set batch size here manually
                         validation_split=0.1, verbose=1,
                         callbacks=[early_stopping, reduce_lr])

# Make predictions with the optimized model
y_pred_lstm = best_model.predict(X_test_seq)
y_pred_lstm = scaler_y.inverse_transform(y_pred_lstm)
y_test_actual = scaler_y.inverse_transform(y_test_seq.reshape(-1, 1)).reshape(y_test_seq.shape)

rmse_best = np.sqrt(mean_squared_error(y_test_actual, y_pred_lstm))
mae_best = mean_absolute_error(y_test_actual, y_pred_lstm)
r2_best = r2_score(y_test_actual, y_pred_lstm)
mse_best = mean_squared_error(y_test_actual, y_pred_lstm)




In [ ]:
sns.set(style="whitegrid")

# Plot Actual vs Predicted
plt.figure(figsize=(14, 6))
plt.plot(y_test_actual.flatten(), label='Actual Production', color='blue', linewidth=2)
plt.plot(y_pred_lstm.flatten(), label='Predicted Production', color='orange', linewidth=2)
plt.xlabel('Time', fontsize=14)
plt.ylabel('Production (MWh)', fontsize=14)
plt.title('Actual vs Predicted Production (LSTM)', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()

# Scatter Plot of Actual vs Predicted Values
plt.figure(figsize=(8, 8))
plt.scatter(y_test_actual.flatten(), y_pred_lstm.flatten(), alpha=0.6, color='mediumorchid', edgecolors='k', s=50)
plt.plot([y_test_actual.min(), y_test_actual.max()], [y_test_actual.min(), y_test_actual.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Production (MWh)', fontsize=14)
plt.ylabel('Predicted Production (MWh)', fontsize=14)
plt.title('Scatter Plot of Actual vs Predicted Production (LSTM)', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()

# Plotting the Error Distribution
plt.figure(figsize=(10, 6))
sns.histplot(errors.flatten(), kde=True, color='forestgreen', bins=30, edgecolor='black')
plt.xlabel('Prediction Error (MWh)', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
plt.title('Error Distribution (LSTM)', fontsize=16)
plt.grid(True)
plt.show()

# Calculate performance metrics and print
rmse_best = np.sqrt(mean_squared_error(y_test_actual, y_pred_lstm))
mae_best = mean_absolute_error(y_test_actual, y_pred_lstm)
r2_best = r2_score(y_test_actual, y_pred_lstm)
mse_best = mean_squared_error(y_test_actual, y_pred_lstm)

print(f"Optimized RMSE: {rmse_best:.4f}, MAE: {mae_best:.4f}, R² Score: {r2_best:.4f}, MSE: {mse_best:.4f}")

### Solar LSTM Model

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras_tuner as kt
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Assuming wind_merged_data is already loaded and available
features = ['ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_SW_DNI', 'ALLSKY_SFC_SW_DIFF', 'WS10M', 'T2M']
target = 'Production'

solar_merged_data = solar_merged_data.dropna()


X = solar_merged_data[features]
y = solar_merged_data[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, shuffle=False)

scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

def create_sequences(features, target, time_steps=12):
    X, y = [], []
    for i in range(len(features) - time_steps):
        X.append(features[i:i + time_steps])
        y.append(target[i + time_steps])
    return np.array(X), np.array(y)

time_steps = 12
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, time_steps)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, time_steps)

def build_model(hp):
    units = hp.Int('units', min_value=50, max_value=100, step=25)
    dropout_rate = hp.Float('dropout_rate', min_value=0.2, max_value=0.3, step=0.1)
    learning_rate = hp.Choice('learning_rate', values=[0.001, 0.01])

    model = Sequential([
        Input(shape=(X_train_seq.shape[1], X_train_seq.shape[2])),
        Bidirectional(LSTM(units, return_sequences=True)),
        Dropout(dropout_rate),
        Bidirectional(LSTM(units, return_sequences=False)),
        Dropout(dropout_rate),
        Dense(1)
    ])

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mean_squared_error')
    return model

# Keras Tuner
tuner = kt.RandomSearch(
    build_model,
    objective='val_loss',
    max_trials=10,
    executions_per_trial=1,
    directory='keras_tuner_dir',
    project_name='lstm_tuning'
)

early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=0.00001)

# Perform the hyperparameter search
tuner.search(X_train_seq, y_train_seq, epochs=15, validation_split=0.1,
             callbacks=[early_stopping, reduce_lr], verbose=2)

# Get the optimal hyperparameters
best_hps = tuner.get_best_hyperparameters(num_trials=1)[0]

print(f"""
The optimal number of units in the LSTM layer is {best_hps.get('units')},
the optimal dropout rate is {best_hps.get('dropout_rate')},
and the optimal learning rate is {best_hps.get('learning_rate')}.
""")

# Build the model with the optimal hyperparameters and train it
best_model = tuner.hypermodel.build(best_hps)

# Manually specify the number of epochs and batch size
history = best_model.fit(X_train_seq, y_train_seq, epochs=15,  # Set epochs here manually
                         batch_size=32,  # Set batch size here manually
                         validation_split=0.1, verbose=1,
                         callbacks=[early_stopping, reduce_lr])

# Make predictions with the optimized model
y_pred_lstm = best_model.predict(X_test_seq)
y_pred_lstm = scaler_y.inverse_transform(y_pred_lstm)
y_test_actual = scaler_y.inverse_transform(y_test_seq.reshape(-1, 1)).reshape(y_test_seq.shape)

rmse_best = np.sqrt(mean_squared_error(y_test_actual, y_pred_lstm))
mae_best = mean_absolute_error(y_test_actual, y_pred_lstm)
r2_best = r2_score(y_test_actual, y_pred_lstm)
mse_best = mean_squared_error(y_test_actual, y_pred_lstm)

print(f"Optimized RMSE: {rmse_best:.4f}, MAE: {mae_best:.4f}, R² Score: {r2_best:.4f}, MSE: {mse_best:.4f}")

sns.set(style="whitegrid")

# Plot Actual vs Predicted
plt.figure(figsize=(14, 6))
plt.plot(y_test_actual.flatten(), label='Actual Production', color='blue', linewidth=2)
plt.plot(y_pred_lstm.flatten(), label='Predicted Production', color='orange', linewidth=2)
plt.xlabel('Time', fontsize=14)
plt.ylabel('Production (MWh)', fontsize=14)
plt.title('Actual vs Predicted Production (LSTM)', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()

# Scatter Plot of Actual vs Predicted Values
plt.figure(figsize=(8, 8))
plt.scatter(y_test_actual.flatten(), y_pred_lstm.flatten(), alpha=0.6, color='mediumorchid', edgecolors='k', s=50)
plt.plot([y_test_actual.min(), y_test_actual.max()], [y_test_actual.min(), y_test_actual.max()], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Production (MWh)', fontsize=14)
plt.ylabel('Predicted Production (MWh)', fontsize=14)
plt.title('Scatter Plot of Actual vs Predicted Production (LSTM)', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()

# Plotting the Error Distribution
plt.figure(figsize=(10, 6))
sns.histplot(errors.flatten(), kde=True, color='forestgreen', bins=30, edgecolor='black')
plt.xlabel('Prediction Error (MWh)', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
plt.title('Error Distribution (LSTM)', fontsize=16)
plt.grid(True)
plt.show()



### Solar Holt-Winters model

In [ ]:
solar_merged_data.dropna()
solar_merged_data.isnull().sum()

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import numpy as np
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
import keras_tuner as kt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Ensure the data is sorted and has a consistent datetime index
solar_merged_data = solar_merged_data.sort_index()
solar_merged_data.index = pd.to_datetime(solar_merged_data.index)
solar_merged_data = solar_merged_data.asfreq('H')  # Set frequency to hourly


target = solar_merged_data['Production']

# Splitting the data
train_data_hw, test_data_hw = train_test_split(target, test_size=0.30, random_state=42, shuffle=False)

train_data_hw = train_data_hw.dropna()
test_data_hw = test_data_hw.dropna()

# Define the parameter grid
param_grid = {
    'trend': ['add'],  # Types of trends
    'seasonal': ['add'],  # Types of seasonality
    'seasonal_periods': [4, 8, 12, 24]  # Different seasonal periods to try
}

# Prepare the grid of parameters
all_params = list(itertools.product(param_grid['trend'], param_grid['seasonal'], param_grid['seasonal_periods']))

# Track the best model and its performance
best_model = None
best_params = None
best_rmse = float('inf')

# Loop through all combinations of hyperparameters
for trend, seasonal, seasonal_periods in all_params:
    try:
        # Fit the Holt-Winters model
        hw_model = ExponentialSmoothing(
            train_data_hw,
            trend=trend,
            seasonal=seasonal,
            seasonal_periods=seasonal_periods,
            initialization_method="estimated"
        ).fit(optimized=True)

        # Make predictions
        y_pred_hw = hw_model.forecast(steps=len(test_data_hw))

        # Align lengths by trimming
        min_len = min(len(y_pred_hw), len(test_data_hw))
        y_pred_hw = y_pred_hw[:min_len]
        test_data_hw_trimmed = test_data_hw[:min_len]

        # Calculate RMSE
        rmse_hw = np.sqrt(mean_squared_error(test_data_hw_trimmed, y_pred_hw))

        # Check if this is the best model so far
        if rmse_hw < best_rmse:
            best_rmse = rmse_hw
            best_model = hw_model
            best_params = (trend, seasonal, seasonal_periods)

        print(f"Tested params - Trend: {trend}, Seasonal: {seasonal}, Seasonal Periods: {seasonal_periods} -> RMSE: {rmse_hw:.4f}")

    except Exception as e:
        print(f"Model failed with parameters Trend: {trend}, Seasonal: {seasonal}, Seasonal Periods: {seasonal_periods}")
        print(f"Error: {e}")

# Check if a best model was found
if best_model is None:
    print("No suitable model was found. Please check the data and hyperparameters.")
else:
    print(f"Best parameters found: Trend: {best_params[0]}, Seasonal: {best_params[1]}, Seasonal Periods: {best_params[2]}")
    print(f"Best RMSE: {best_rmse:.4f}")

    # Use the best model to make predictions
    y_pred_hw = best_model.forecast(steps=len(test_data_hw))

    # Align lengths by trimming for final metrics calculation
    min_len = min(len(y_pred_hw), len(test_data_hw))
    y_pred_hw = y_pred_hw[:min_len]
    test_data_hw_trimmed = test_data_hw[:min_len]

    rmse_best = np.sqrt(mean_squared_error(test_data_hw_trimmed, y_pred_hw))
    mae_best = mean_absolute_error(test_data_hw_trimmed, y_pred_hw)
    r2_best = r2_score(test_data_hw_trimmed, y_pred_hw)
    mse_best = mean_squared_error(test_data_hw_trimmed, y_pred_hw)
    
    print(f"Optimized RMSE: {rmse_best:.4f}, MAE: {mae_best:.4f}, R² Score: {r2_best:.4f}, MSE: {mse_best:.4f}")

    # Plot results using the best model
    plt.figure(figsize=(14, 5))
    plt.plot(test_data_hw_trimmed.index, test_data_hw_trimmed, label='Actual Production', color='blue')
    plt.plot(test_data_hw_trimmed.index, y_pred_hw, label='Predicted Production (Holt-Winters)', color='orange', alpha=0.7)
    plt.xlabel('Time')
    plt.ylabel('Production')
    plt.title('Actual vs Predicted Production (Holt-Winters)')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Scatter Plot of Actual vs. Predicted Values
    plt.figure(figsize=(8, 8))
    plt.scatter(test_data_hw_trimmed, y_pred_hw, alpha=0.5, color='purple')
    plt.xlabel('Actual Production')
    plt.ylabel('Predicted Production')
    plt.title('Actual vs Predicted Production (Holt-Winters)')
    plt.plot([test_data_hw_trimmed.min(), test_data_hw_trimmed.max()], [test_data_hw_trimmed.min(), test_data_hw_trimmed.max()], 'r--')
    plt.grid(True)
    plt.show()

    # Plotting the Error Distribution
    errors = test_data_hw_trimmed - y_pred_hw
    plt.figure(figsize=(10, 6))
    sns.histplot(errors, kde=True, color='green')
    plt.xlabel('Prediction Error')
    plt.title('Error Distribution')
    plt.grid(True)
    plt.show()


### Wind Holt-Winters Model

In [ ]:
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import itertools
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure the data is sorted and has a consistent datetime index
wind_merged_data = wind_merged_data.sort_index()
wind_merged_data.index = pd.to_datetime(wind_merged_data.index)
wind_merged_data = wind_merged_data.asfreq('H')  # Set frequency to hourly


target = wind_merged_data['Production']

# Splitting the data
train_data_hw, test_data_hw = train_test_split(target, test_size=0.30, random_state=42, shuffle=False)

train_data_hw = train_data_hw.dropna()
test_data_hw = test_data_hw.dropna()

# Define the parameter grid
param_grid = {
    'trend': ['add'],  # Types of trends
    'seasonal': ['add'],  # Types of seasonality
    'seasonal_periods': [4, 8, 12, 24]  # Different seasonal periods to try
}

# Prepare the grid of parameters
all_params = list(itertools.product(param_grid['trend'], param_grid['seasonal'], param_grid['seasonal_periods']))

# Track the best model and its performance
best_model = None
best_params = None
best_rmse = float('inf')

# Loop through all combinations of hyperparameters
for trend, seasonal, seasonal_periods in all_params:
    try:
        # Fit the Holt-Winters model
        hw_model = ExponentialSmoothing(
            train_data_hw,
            trend=trend,
            seasonal=seasonal,
            seasonal_periods=seasonal_periods,
            initialization_method="estimated"
        ).fit(optimized=True)

        # Make predictions
        y_pred_hw = hw_model.forecast(steps=len(test_data_hw))

        # Align lengths by trimming
        min_len = min(len(y_pred_hw), len(test_data_hw))
        y_pred_hw = y_pred_hw[:min_len]
        test_data_hw_trimmed = test_data_hw[:min_len]

        # Calculate RMSE
        rmse_hw = np.sqrt(mean_squared_error(test_data_hw_trimmed, y_pred_hw))

        # Check if this is the best model so far
        if rmse_hw < best_rmse:
            best_rmse = rmse_hw
            best_model = hw_model
            best_params = (trend, seasonal, seasonal_periods)

        print(f"Tested params - Trend: {trend}, Seasonal: {seasonal}, Seasonal Periods: {seasonal_periods} -> RMSE: {rmse_hw:.4f}")

    except Exception as e:
        print(f"Model failed with parameters Trend: {trend}, Seasonal: {seasonal}, Seasonal Periods: {seasonal_periods}")
        print(f"Error: {e}")

# Check if a best model was found
if best_model is None:
    print("No suitable model was found. Please check the data and hyperparameters.")
else:
    print(f"Best parameters found: Trend: {best_params[0]}, Seasonal: {best_params[1]}, Seasonal Periods: {best_params[2]}")
    print(f"Best RMSE: {best_rmse:.4f}")

    # Use the best model to make predictions
    y_pred_hw = best_model.forecast(steps=len(test_data_hw))

    # Align lengths by trimming for final metrics calculation
    min_len = min(len(y_pred_hw), len(test_data_hw))
    y_pred_hw = y_pred_hw[:min_len]
    test_data_hw_trimmed = test_data_hw[:min_len]

    rmse_best = np.sqrt(mean_squared_error(test_data_hw_trimmed, y_pred_hw))
    mae_best = mean_absolute_error(test_data_hw_trimmed, y_pred_hw)
    r2_best = r2_score(test_data_hw_trimmed, y_pred_hw)
    mse_best = mean_squared_error(test_data_hw_trimmed, y_pred_hw)
    
    print(f"Optimized RMSE: {rmse_best:.4f}, MAE: {mae_best:.4f}, R² Score: {r2_best:.4f}, MSE: {mse_best:.4f}")

    # Plot results using the best model
    plt.figure(figsize=(14, 5))
    plt.plot(test_data_hw_trimmed.index, test_data_hw_trimmed, label='Actual Production', color='blue')
    plt.plot(test_data_hw_trimmed.index, y_pred_hw, label='Predicted Production (Holt-Winters)', color='orange', alpha=0.7)
    plt.xlabel('Time')
    plt.ylabel('Production')
    plt.title('Actual vs Predicted Production (Holt-Winters)')
    plt.legend()
    plt.grid(True)
    plt.show()

    # Scatter Plot of Actual vs. Predicted Values
    plt.figure(figsize=(8, 8))
    plt.scatter(test_data_hw_trimmed, y_pred_hw, alpha=0.5, color='purple')
    plt.xlabel('Actual Production')
    plt.ylabel('Predicted Production')
    plt.title('Actual vs Predicted Production (Holt-Winters)')
    plt.plot([test_data_hw_trimmed.min(), test_data_hw_trimmed.max()], [test_data_hw_trimmed.min(), test_data_hw_trimmed.max()], 'r--')
    plt.grid(True)
    plt.show()

    # Plotting the Error Distribution
    errors = test_data_hw_trimmed - y_pred_hw
    plt.figure(figsize=(10, 6))
    sns.histplot(errors, kde=True, color='green')
    plt.xlabel('Prediction Error')
    plt.title('Error Distribution')
    plt.grid(True)
    plt.show()


### Solar Facebook Prophet

In [ ]:
from prophet import Prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import ParameterGrid
from prophet.diagnostics import cross_validation, performance_metrics
from sklearn.model_selection import train_test_split

df_prophet = solar_merged_data.reset_index()[['timestamp', 'Production']].rename(columns={'timestamp': 'ds', 'Production': 'y'})

# Remove timezone information
df_prophet['ds'] = df_prophet['ds'].dt.tz_localize(None)

# Split the data into train and test sets
train_data_prophet, test_data_prophet = train_test_split(df_prophet, test_size=0.30, random_state=42, shuffle=False)


# Function to calculate MAPE
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / np.where(y_true != 0, y_true, np.finfo(float).eps))) * 100

# Define the hyperparameter grid
param_grid = {
    'changepoint_prior_scale': [0.1, 0.5],
    'seasonality_prior_scale': [0.1, 1.0, 3.0],
    'seasonality_mode': ['additive', 'multiplicative']
}

# Track the best model and its performance
best_model = None
best_params = None
best_rmse = float('inf')

# Loop through all combinations of hyperparameters
for params in ParameterGrid(param_grid):
    print(f"Testing parameters: {params}")
    prophet_model = Prophet(
        changepoint_prior_scale=params['changepoint_prior_scale'],
        seasonality_prior_scale=params['seasonality_prior_scale'],
        seasonality_mode=params['seasonality_mode'],
        daily_seasonality=True
    )
    
    prophet_model.fit(train_data_prophet)
    future = prophet_model.make_future_dataframe(periods=len(test_data_prophet), freq='H')
    forecast = prophet_model.predict(future)
    
    y_pred = forecast['yhat'].iloc[-len(test_data_prophet):].values
    y_test = test_data_prophet['y'].values
    
    mask = ~np.isnan(y_test)
    y_test_cleaned = y_test[mask]
    y_pred_cleaned = y_pred[mask]
    
    rmse_best = np.sqrt(mean_squared_error(y_test_cleaned, y_pred_cleaned))
    mae_best = mean_absolute_error(y_test_cleaned, y_pred_cleaned)
    r2_best = r2_score(y_test_cleaned, y_pred_cleaned)
    mse_best = mean_squared_error(y_test_cleaned, y_pred_cleaned)
    
    print(f"Optimized RMSE: {rmse_best:.4f}, MAE: {mae_best:.4f}, R² Score: {r2_best:.4f}, MSE: {mse_best:.4f}")
    
    # Track the best model
    if rmse_best < best_rmse:
        best_rmse = rmse_best
        best_model = prophet_model
        best_params = params

print(f"\nBest parameters found: {best_params}")
print(f"Best RMSE: {best_rmse:.4f}")

# Use the best model to make predictions
future = best_model.make_future_dataframe(periods=len(test_data_prophet), freq='H')
forecast = best_model.predict(future)

y_pred_prophet = forecast['yhat'].iloc[-len(test_data_prophet):].values
y_test_actual = test_data_prophet['y'].values

# Clean NaN values
mask = ~np.isnan(y_test_actual)
y_test_actual_cleaned = y_test_actual[mask]
y_pred_prophet_cleaned = y_pred_prophet[mask]


df_cv = cross_validation(best_model, initial='90 days', period='40 days', horizon='365 days')
df_p = performance_metrics(df_cv)

print(df_p.head())

rmse_best = np.sqrt(mean_squared_error(y_test_cleaned, y_pred_cleaned))
mae_best = mean_absolute_error(y_test_cleaned, y_pred_cleaned)
r2_best = r2_score(y_test_cleaned, y_pred_cleaned)
mse_best = mean_squared_error(y_test_cleaned, y_pred_cleaned)

# Set a seaborn style for consistent aesthetics
sns.set(style="whitegrid")

# Plot Actual vs Predicted Production (Prophet)
plt.figure(figsize=(14, 6))
plt.plot(y_test_actual_cleaned, label='Actual Production', color='blue', linewidth=2)
plt.plot(y_pred_prophet_cleaned, label='Predicted Production (Prophet)', color='orange', linewidth=2, alpha=0.6)
plt.xlabel('Time', fontsize=14)
plt.ylabel('Production (MWh)', fontsize=14)
plt.title('Actual vs Predicted Production (Prophet)', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()
plt.savefig('actual_vs_predicted_prophet.png')


# Scatter Plot of Actual vs Predicted Production
plt.figure(figsize=(8, 8))
plt.scatter(y_test_actual_cleaned, y_pred_prophet_cleaned, alpha=0.6, color='mediumorchid', edgecolors='k', s=50)
plt.plot([y_test_actual_cleaned.min(), y_test_actual_cleaned.max()], [y_test_actual_cleaned.min(), y_test_actual_cleaned.max()], 'r--', lw=2, label='Ideal Prediction')
plt.xlabel('Actual Production (MWh)', fontsize=14)
plt.ylabel('Predicted Production (MWh)', fontsize=14)
plt.title('Scatter Plot of Actual vs Predicted Production (Prophet)', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()
plt.savefig('scatter_actual_vs_predicted_prophet.png', dpi=300, bbox_inches='tight')

# Display the best model metrics
rmse_best = np.sqrt(mean_squared_error(y_test_actual_cleaned, y_pred_prophet_cleaned))
mae_best = mean_absolute_error(y_test_actual_cleaned, y_pred_prophet_cleaned)
r2_best = r2_score(y_test_actual_cleaned, y_pred_prophet_cleaned)
mse_best = mean_squared_error(y_test_actual_cleaned, y_pred_prophet_cleaned)

print(f"Optimized RMSE: {rmse_best:.4f}, MAE: {mae_best:.4f}, R² Score: {r2_best:.4f}, MSE: {mse_best:.4f}")

### Wind Facebook Prophet

In [ ]:
df_prophet = wind_merged_data.reset_index()[['timestamp', 'Production']].rename(columns={'timestamp': 'ds', 'Production': 'y'})

# Remove timezone information
df_prophet['ds'] = df_prophet['ds'].dt.tz_localize(None)

# Split the data into train and test sets
train_data_prophet, test_data_prophet = train_test_split(df_prophet, test_size=0.30, random_state=42, shuffle=False)


# Function to calculate MAPE
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / np.where(y_true != 0, y_true, np.finfo(float).eps))) * 100

# Define the hyperparameter grid
param_grid = {
    'changepoint_prior_scale': [0.1, 0.5],
    'seasonality_prior_scale': [0.1, 1.0, 3.0],
    'seasonality_mode': ['additive', 'multiplicative']
}

# Track the best model and its performance
best_model = None
best_params = None
best_rmse = float('inf')

# Loop through all combinations of hyperparameters
for params in ParameterGrid(param_grid):
    print(f"Testing parameters: {params}")
    prophet_model = Prophet(
        changepoint_prior_scale=params['changepoint_prior_scale'],
        seasonality_prior_scale=params['seasonality_prior_scale'],
        seasonality_mode=params['seasonality_mode'],
        daily_seasonality=True
    )
    
    prophet_model.fit(train_data_prophet)
    future = prophet_model.make_future_dataframe(periods=len(test_data_prophet), freq='H')
    forecast = prophet_model.predict(future)
    
    y_pred = forecast['yhat'].iloc[-len(test_data_prophet):].values
    y_test = test_data_prophet['y'].values
    
    mask = ~np.isnan(y_test)
    y_test_cleaned = y_test[mask]
    y_pred_cleaned = y_pred[mask]
    
    rmse_best = np.sqrt(mean_squared_error(y_test_cleaned, y_pred_cleaned))
    mae_best = mean_absolute_error(y_test_cleaned, y_pred_cleaned)
    r2_best = r2_score(y_test_cleaned, y_pred_cleaned)
    mse_best = mean_squared_error(y_test_cleaned, y_pred_cleaned)
    
    print(f"Optimized RMSE: {rmse_best:.4f}, MAE: {mae_best:.4f}, R² Score: {r2_best:.4f}, MSE: {mse_best:.4f}")
    
    # Track the best model
    if rmse_best < best_rmse:
        best_rmse = rmse_best
        best_model = prophet_model
        best_params = params

print(f"\nBest parameters found: {best_params}")
print(f"Best RMSE: {best_rmse:.4f}")

# Use the best model to make predictions
future = best_model.make_future_dataframe(periods=len(test_data_prophet), freq='H')
forecast = best_model.predict(future)

y_pred_prophet = forecast['yhat'].iloc[-len(test_data_prophet):].values
y_test_actual = test_data_prophet['y'].values

# Clean NaN values
mask = ~np.isnan(y_test_actual)
y_test_actual_cleaned = y_test_actual[mask]
y_pred_prophet_cleaned = y_pred_prophet[mask]


df_cv = cross_validation(best_model, initial='90 days', period='40 days', horizon='365 days')
df_p = performance_metrics(df_cv)

print(df_p.head())

rmse_best = np.sqrt(mean_squared_error(y_test_cleaned, y_pred_cleaned))
mae_best = mean_absolute_error(y_test_cleaned, y_pred_cleaned)
r2_best = r2_score(y_test_cleaned, y_pred_cleaned)
mse_best = mean_squared_error(y_test_cleaned, y_pred_cleaned)

# Set a seaborn style for consistent aesthetics
sns.set(style="whitegrid")

# Plot Actual vs Predicted Production (Prophet)
plt.figure(figsize=(14, 6))
plt.plot(y_test_actual_cleaned, label='Actual Production', color='blue', linewidth=2)
plt.plot(y_pred_prophet_cleaned, label='Predicted Production (Prophet)', color='orange', linewidth=2, alpha=0.6)
plt.xlabel('Time', fontsize=14)
plt.ylabel('Production (MWh)', fontsize=14)
plt.title('Actual vs Predicted Production (Prophet)', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()
plt.savefig('actual_vs_predicted_prophet.png')


# Scatter Plot of Actual vs Predicted Production
plt.figure(figsize=(8, 8))
plt.scatter(y_test_actual_cleaned, y_pred_prophet_cleaned, alpha=0.6, color='mediumorchid', edgecolors='k', s=50)
plt.plot([y_test_actual_cleaned.min(), y_test_actual_cleaned.max()], [y_test_actual_cleaned.min(), y_test_actual_cleaned.max()], 'r--', lw=2, label='Ideal Prediction')
plt.xlabel('Actual Production (MWh)', fontsize=14)
plt.ylabel('Predicted Production (MWh)', fontsize=14)
plt.title('Scatter Plot of Actual vs Predicted Production (Prophet)', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()
plt.savefig('scatter_actual_vs_predicted_prophet.png', dpi=300, bbox_inches='tight')

# Display the best model metrics
rmse_best = np.sqrt(mean_squared_error(y_test_actual_cleaned, y_pred_prophet_cleaned))
mae_best = mean_absolute_error(y_test_actual_cleaned, y_pred_prophet_cleaned)
r2_best = r2_score(y_test_actual_cleaned, y_pred_prophet_cleaned)
mse_best = mean_squared_error(y_test_actual_cleaned, y_pred_prophet_cleaned)

print(f"Optimized RMSE: {rmse_best:.4f}, MAE: {mae_best:.4f}, R² Score: {r2_best:.4f}, MSE: {mse_best:.4f}")

### Solar Ensemble Method (SVM, LSTM, Prophet)

In [ ]:
solar_merged_data = solar_merged_data.dropna()
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional, Input
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import Sequential
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np
import itertools
import matplotlib.pyplot as plt
import seaborn as sns
from prophet import Prophet
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVR


# Prophet Model
df_prophet = solar_merged_data.reset_index()[['timestamp', 'Production']].rename(columns={'timestamp': 'ds', 'Production': 'y'})
df_prophet['ds'] = df_prophet['ds'].dt.tz_localize(None)
train_data_prophet, test_data_prophet = train_test_split(df_prophet, test_size=0.30, random_state=42, shuffle=False)

prophet_model = Prophet(
        changepoint_prior_scale=0.1,
        seasonality_prior_scale=0.1,
        seasonality_mode='additive',
        daily_seasonality=True
    )
prophet_model.fit(train_data_prophet)
future = prophet_model.make_future_dataframe(periods=len(test_data_prophet), freq='H')
forecast = prophet_model.predict(future)
y_pred_prophet = forecast['yhat'].iloc[-len(test_data_prophet):].values
y_test_actual_prophet = test_data_prophet['y'].values

# Mask out NaNs
mask = ~np.isnan(y_test_actual_prophet)
y_test_actual_cleaned = y_test_actual_prophet[mask]
y_pred_prophet_cleaned = y_pred_prophet[mask]

def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    return np.mean(np.abs((y_true - y_pred) / np.maximum(np.abs(y_true), 1e-10))) * 100

rmse_prophet = np.sqrt(mean_squared_error(y_test_actual_cleaned, y_pred_prophet_cleaned))
mae_prophet = mean_absolute_error(y_test_actual_cleaned, y_pred_prophet_cleaned)
r2_prophet = r2_score(y_test_actual_cleaned, y_pred_prophet_cleaned)
mse_prophet = mean_squared_error(y_test_actual_cleaned, y_pred_prophet_cleaned)

print(f"Optimized RMSE: {rmse_prophet:.4f}, MAE: {mae_prophet:.4f}, R² Score: {r2_prophet:.4f}, MSE: {mse_prophet:.4f}")

# LSTM Model
features = ['ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_SW_DNI', 'ALLSKY_SFC_SW_DIFF', 'ALLSKY_KT', 'WS10M', 'T2M']
target = 'Production'

X = solar_merged_data[features]
y = solar_merged_data[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, shuffle=False)

scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

def create_sequences(features, target, time_steps=24):
    X, y = [], []
    for i in range(len(features) - time_steps):
        X.append(features[i:i+time_steps])
        y.append(target[i+time_steps])
    return np.array(X), np.array(y)

time_steps = 24
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, time_steps)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, time_steps)

lstm_model = Sequential()
lstm_model.add(Bidirectional(LSTM(75, return_sequences=True, input_shape=(X_train_seq.shape[1], X_train_seq.shape[2]))))
lstm_model.add(Dropout(0.2))
lstm_model.add(Bidirectional(LSTM(75, return_sequences=False)))
lstm_model.add(Dropout(0.2))
lstm_model.add(Dense(1))

lstm_model.compile(optimizer='adam', loss='mean_squared_error')
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = lstm_model.fit(X_train_seq, y_train_seq, epochs=20, batch_size=32, validation_split=0.1, verbose=1, callbacks=[early_stopping])

y_pred_lstm = lstm_model.predict(X_test_seq)
y_pred_lstm = scaler_y.inverse_transform(y_pred_lstm)
y_test_actual_lstm = scaler_y.inverse_transform(y_test_seq.reshape(-1, 1)).reshape(y_test_seq.shape)

rmse_lstm = np.sqrt(mean_squared_error(y_test_actual_lstm, y_pred_lstm))
mae_lstm = mean_absolute_error(y_test_actual_lstm, y_pred_lstm)
r2_lstm = r2_score(y_test_actual_lstm, y_pred_lstm)
mse_lstm = mean_squared_error(y_test_actual_lstm, y_pred_lstm)
print(f"Optimized RMSE: {rmse_lstm:.4f}, MAE: {mae_lstm:.4f}, R² Score: {r2_lstm:.4f}, MSE: {mse_lstm:.4f}")

# SVM Model
svm_model = SVR(kernel='rbf', C=1000, epsilon=0.2)
svm_model.fit(X_train_scaled, y_train_scaled.ravel())

y_pred_svm = svm_model.predict(X_test_scaled)
y_pred_svm = scaler_y.inverse_transform(y_pred_svm.reshape(-1, 1)).flatten()

rmse_svm = np.sqrt(mean_squared_error(y_test_scaled, y_pred_svm))
mae_svm = mean_absolute_error(y_test_scaled, y_pred_svm)
r2_svm = r2_score(y_test_scaled, y_pred_svm)
mse_svm = mean_squared_error(y_test_scaled, y_pred_svm)

print(f"Optimized RMSE: {rmse_svm:.4f}, MAE: {mae_svm:.4f}, R² Score: {r2_svm:.4f}, MSE: {mse_svm:.4f}")

# Trim predictions to the minimum length
min_length = min(len(y_pred_prophet_cleaned), len(y_pred_lstm.flatten()), len(y_pred_svm))

# Align all predictions to the same length
y_pred_prophet_trimmed = y_pred_prophet_cleaned[:min_length]
y_pred_lstm_trimmed = y_pred_lstm.flatten()[:min_length]
y_pred_svm_trimmed = y_pred_svm[:min_length]

# Corresponding y_test_actual_trimmed
y_test_actual_trimmed = y_test_actual_cleaned[:min_length]

# Weighted Ensemble
weight_prophet = 0.1
weight_lstm = 0.4
weight_svm = 0.5


y_pred_ensemble = (weight_prophet * y_pred_prophet_trimmed + 
                   weight_lstm * y_pred_lstm_trimmed + 
                   weight_svm * y_pred_svm_trimmed)

# Calculate ensemble metrics
rmse_best = np.sqrt(mean_squared_error(y_test_actual_trimmed, y_pred_ensemble))
mae_best = mean_absolute_error(y_test_actual_trimmed, y_pred_ensemble)
r2_best = r2_score(y_test_actual_trimmed, y_pred_ensemble)
mse_best = mean_squared_error(y_test_actual_trimmed, y_pred_ensemble)

print(f"Optimized RMSE: {rmse_best:.4f}, MAE: {mae_best:.4f}, R² Score: {r2_best:.4f}, MSE: {mse_best:.4f}")

# Visualization
# Set a seaborn style for consistent aesthetics
sns.set(style="whitegrid")

# Actual vs Predicted Production (Weighted Ensemble)
plt.figure(figsize=(14, 6))
plt.plot(y_test_actual_trimmed.flatten(), label='Actual Production', color='blue', linewidth=2)
plt.plot(y_pred_ensemble.flatten(), label='Predicted Production (Weighted Ensemble)', color='orange', linewidth=2)
plt.xlabel('Time', fontsize=14)
plt.ylabel('Production (MWh)', fontsize=14)
plt.title('Actual vs Predicted Production (Weighted Ensemble)', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()

# Scatter Plot of Actual vs Predicted Production (Weighted Ensemble)
plt.figure(figsize=(8, 8))
plt.scatter(y_test_actual_trimmed.flatten(), y_pred_ensemble.flatten(), alpha=0.6, color='mediumorchid', edgecolors='k', s=50)
plt.plot([y_test_actual_trimmed.min(), y_test_actual_trimmed.max()], [y_test_actual_trimmed.min(), y_test_actual_trimmed.max()], 'r--', lw=2, label='Ideal Prediction')
plt.xlabel('Actual Production (MWh)', fontsize=14)
plt.ylabel('Predicted Production (MWh)', fontsize=14)
plt.title('Scatter Plot of Actual vs Predicted Production (Weighted Ensemble)', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()

# Plotting the Error Distribution (Weighted Ensemble)
errors_ensemble = y_test_actual_trimmed - y_pred_ensemble
plt.figure(figsize=(10, 6))
sns.histplot(errors_ensemble.flatten(), kde=True, color='forestgreen', bins=30, edgecolor='black')
plt.xlabel('Prediction Error (MWh)', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
plt.title('Error Distribution (Weighted Ensemble)', fontsize=16)
plt.grid(True)
plt.show()


### Wind Ensemble Method (SVM, LSTM, Prophet)

In [ ]:
wind_merged_data = wind_merged_data.dropna()

# Prophet Model
df_prophet = wind_merged_data.reset_index()[['timestamp', 'Production']].rename(columns={'timestamp': 'ds', 'Production': 'y'})
df_prophet['ds'] = df_prophet['ds'].dt.tz_localize(None)
train_data_prophet, test_data_prophet = train_test_split(df_prophet, test_size=0.30, random_state=42, shuffle=False)

prophet_model = Prophet(
        changepoint_prior_scale=0.1,
        seasonality_prior_scale=0.1,
        seasonality_mode='additive',
        daily_seasonality=True
    )
prophet_model.fit(train_data_prophet)
future = prophet_model.make_future_dataframe(periods=len(test_data_prophet), freq='H')
forecast = prophet_model.predict(future)
y_pred_prophet = forecast['yhat'].iloc[-len(test_data_prophet):].values
y_test_actual_prophet = test_data_prophet['y'].values

# Mask out NaNs
mask = ~np.isnan(y_test_actual_prophet)
y_test_actual_cleaned = y_test_actual_prophet[mask]
y_pred_prophet_cleaned = y_pred_prophet[mask]


rmse_prophet = np.sqrt(mean_squared_error(y_test_actual_cleaned, y_pred_prophet_cleaned))
mae_prophet = mean_absolute_error(y_test_actual_cleaned, y_pred_prophet_cleaned)
r2_prophet = r2_score(y_test_actual_cleaned, y_pred_prophet_cleaned)
mse_prophet = mean_squared_error(y_test_actual_cleaned, y_pred_prophet_cleaned)

print(f"Prophet RMSE: {rmse_prophet:.4f}, MAE: {mae_prophet:.4f}, R² Score: {r2_prophet:.4f}, MSE: {mse_prophet:.4f}")

# LSTM Model
features = ['ALLSKY_SFC_SW_DWN', 'ALLSKY_SFC_SW_DNI', 'ALLSKY_SFC_SW_DIFF', 'ALLSKY_KT', 'WS10M', 'T2M']
target = 'Production'

X = solar_merged_data[features]
y = solar_merged_data[target]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42, shuffle=False)

scaler_X = MinMaxScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = MinMaxScaler()
y_train_scaled = scaler_y.fit_transform(y_train.values.reshape(-1, 1))
y_test_scaled = scaler_y.transform(y_test.values.reshape(-1, 1))

def create_sequences(features, target, time_steps=24):
    X, y = [], []
    for i in range(len(features) - time_steps):
        X.append(features[i:i+time_steps])
        y.append(target[i+time_steps])
    return np.array(X), np.array(y)

time_steps = 24
X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train_scaled, time_steps)
X_test_seq, y_test_seq = create_sequences(X_test_scaled, y_test_scaled, time_steps)

lstm_model = Sequential()
lstm_model.add(Bidirectional(LSTM(75, return_sequences=True, input_shape=(X_train_seq.shape[1], X_train_seq.shape[2]))))
lstm_model.add(Dropout(0.2))
lstm_model.add(Bidirectional(LSTM(75, return_sequences=False)))
lstm_model.add(Dropout(0.2))
lstm_model.add(Dense(1))

lstm_model.compile(optimizer='adam', loss='mean_squared_error')
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = lstm_model.fit(X_train_seq, y_train_seq, epochs=20, batch_size=32, validation_split=0.1, verbose=1, callbacks=[early_stopping])

y_pred_lstm = lstm_model.predict(X_test_seq)
y_pred_lstm = scaler_y.inverse_transform(y_pred_lstm)
y_test_actual_lstm = scaler_y.inverse_transform(y_test_seq.reshape(-1, 1)).reshape(y_test_seq.shape)

rmse_lstm = np.sqrt(mean_squared_error(y_test_actual_lstm, y_pred_lstm))
mae_lstm = mean_absolute_error(y_test_actual_lstm, y_pred_lstm)
r2_lstm = r2_score(y_test_actual_lstm, y_pred_lstm)
mse_lstm = mean_squared_error(y_test_actual_lstm, y_pred_lstm)
print(f"LSTM RMSE: {rmse_lstm:.4f}, MAE: {mae_lstm:.4f}, R² Score: {r2_lstm:.4f}, MSE: {mse_lstm:.4f}")

# SVM Model
svm_model = SVR(kernel='rbf', C=1000, epsilon=0.2)
svm_model.fit(X_train_scaled, y_train_scaled.ravel())

y_pred_svm = svm_model.predict(X_test_scaled)
y_pred_svm = scaler_y.inverse_transform(y_pred_svm.reshape(-1, 1)).flatten()

rmse_svm = np.sqrt(mean_squared_error(y_test_scaled, y_pred_svm))
mae_svm = mean_absolute_error(y_test_scaled, y_pred_svm)
r2_svm = r2_score(y_test_scaled, y_pred_svm)
mse_svm = mean_squared_error(y_test_scaled, y_pred_svm)

print(f"SVM RMSE: {rmse_svm:.4f}, MAE: {mae_svm:.4f}, R² Score: {r2_svm:.4f}, MSE: {mse_svm:.4f}")

# Trim predictions to the minimum length
min_length = min(len(y_pred_prophet_cleaned), len(y_pred_lstm.flatten()), len(y_pred_svm))

# Align all predictions to the same length
y_pred_prophet_trimmed = y_pred_prophet_cleaned[:min_length]
y_pred_lstm_trimmed = y_pred_lstm.flatten()[:min_length]
y_pred_svm_trimmed = y_pred_svm[:min_length]

# Corresponding y_test_actual_trimmed
y_test_actual_trimmed = y_test_actual_cleaned[:min_length]

# Weighted Ensemble
weight_prophet = 0.1
weight_lstm = 0.4
weight_svm = 0.5

y_pred_ensemble = (weight_prophet * y_pred_prophet_trimmed + 
                   weight_lstm * y_pred_lstm_trimmed + 
                   weight_svm * y_pred_svm_trimmed)

# Calculate ensemble metrics
rmse_best = np.sqrt(mean_squared_error(y_test_actual_trimmed, y_pred_ensemble))
mae_best = mean_absolute_error(y_test_actual_trimmed, y_pred_ensemble)
r2_best = r2_score(y_test_actual_trimmed, y_pred_ensemble)
mse_best = mean_squared_error(y_test_actual_trimmed, y_pred_ensemble)

print(f"Ensemble RMSE: {rmse_best:.4f}, MAE: {mae_best:.4f}, R² Score: {r2_best:.4f}, MSE: {mse_best:.4f}")

# Visualization
# Set a seaborn style for consistent aesthetics

# Actual vs Predicted Production (Weighted Ensemble)
plt.figure(figsize=(14, 6))
plt.plot(y_test_actual_trimmed.flatten(), label='Actual Production', color='blue', linewidth=2)
plt.plot(y_pred_ensemble.flatten(), label='Predicted Production (Weighted Ensemble)', color='orange', linewidth=2)
plt.xlabel('Time', fontsize=14)
plt.ylabel('Production (MWh)', fontsize=14)
plt.title('Actual vs Predicted Production (Weighted Ensemble)', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()

# Scatter Plot of Actual vs Predicted Production (Weighted Ensemble)
plt.figure(figsize=(8, 8))
plt.scatter(y_test_actual_trimmed.flatten(), y_pred_ensemble.flatten(), alpha=0.6, color='mediumorchid', edgecolors='k', s=50)
plt.plot([y_test_actual_trimmed.min(), y_test_actual_trimmed.max()], [y_test_actual_trimmed.min(), y_test_actual_trimmed.max()], 'r--', lw=2, label='Ideal Prediction')
plt.xlabel('Actual Production (MWh)', fontsize=14)
plt.ylabel('Predicted Production (MWh)', fontsize=14)
plt.title('Scatter Plot of Actual vs Predicted Production (Weighted Ensemble)', fontsize=16)
plt.legend(fontsize=12)
plt.grid(True)
plt.show()

# Plotting the Error Distribution (Weighted Ensemble)
errors_ensemble = y_test_actual_trimmed - y_pred_ensemble
plt.figure(figsize=(10, 6))
sns.histplot(errors_ensemble.flatten(), kde=True, color='forestgreen', bins=30, edgecolor='black')
plt.xlabel('Prediction Error (MWh)', fontsize=14)
plt.ylabel('Frequency', fontsize=14)
plt.title('Error Distribution (Weighted Ensemble)', fontsize=16)
plt.grid(True)
plt.show()
